# Theory of Mind (ToM) Steering Vectors for Gemma-3-4B

This notebook trains steering vectors to enhance Theory of Mind capabilities in Gemma-3-4B using the repeng library.

**What this does:**
- Trains control vectors that steer the model toward better understanding of mental states, beliefs, and intentions
- Exports vectors in `.gguf` format for later use
- Provides both a general ToM vector and specialized vectors for specific ToM skills

**Repository:** https://github.com/ChuloIva/Cogni_map

## 1. Setup & Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone the Cogni_map repository
!git clone https://github.com/ChuloIva/Cogni_map.git
%cd Cogni_map

In [ ]:
# Install dependencies first
!pip install -q transformers torch accelerate sentencepiece

# Install build tools for repeng
!pip install -q hatchling

# Install the repeng library from the cloned repo
# Method 1: Try pip install with pyproject.toml
!pip install -q ToM/repeng/

# Method 2: If above fails, add to Python path directly
import sys
sys.path.insert(0, '/content/Cogni_map/ToM/repeng')

# Verify installation
try:
    from repeng import ControlVector, ControlModel, DatasetEntry
    print("✓ repeng successfully imported!")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    print("\nTrying alternative installation...")
    !pip install -q numpy>=1.26.4 scikit-learn>=1.4.0 tqdm>=4.66.1 gguf>=0.13.0
    sys.path.insert(0, '/content/Cogni_map/ToM/repeng')
    from repeng import ControlVector, ControlModel, DatasetEntry
    print("✓ repeng imported via sys.path!")

In [ ]:
# Optional: Mount Google Drive to save vectors permanently
from google.colab import drive
drive.mount('/content/drive')

# Create directory for saving vectors
!mkdir -p /content/drive/MyDrive/tom_steering_vectors

## 2. Load Model (Gemma-3-4B, Text-Only)

In [ ]:
import json
import torch
import sys
from transformers import AutoModelForCausalLM, AutoConfig, Gemma3ForCausalLM, AutoTokenizer

# Ensure repeng is in path (in case previous cell was skipped)
if '/content/Cogni_map/ToM/repeng' not in sys.path:
    sys.path.insert(0, '/content/Cogni_map/ToM/repeng')

from repeng import ControlVector, ControlModel, DatasetEntry

print("✓ All imports successful!")

In [ ]:
# Model configuration
model_name = "google/gemma-3-4b-it"

print(f"Loading {model_name}...")

# Load config to check if it's a vision-language model
config = AutoConfig.from_pretrained(model_name)

# IMPORTANT: Use bfloat16 instead of float16 to avoid NaN in deeper layers
# bfloat16 has better numerical stability for Gemma models
print("Using bfloat16 for better numerical stability...")

# Load model (skip vision tower if present)
if hasattr(config, 'vision_config'):
    print("Detected vision-language model. Loading text-only version (skipping vision tower)...")
    base_model = Gemma3ForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,  # Changed from float16 to bfloat16
        device_map="auto"
    )
else:
    print("Loading standard causal LM...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,  # Changed from float16 to bfloat16
        device_map="auto"
    )

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = 0  # Set padding token

print("Model loaded successfully!")
print(f"Device: {base_model.device}")
print(f"Dtype: {base_model.dtype}")

In [ ]:
# Wrap with ControlModel for steering
# CRITICAL FIX: Gemma-3-4B-IT is multimodal with layers at model.language_model.layers
# We need to tell repeng where the text layers are by setting repeng_layers

print("Setting up ControlModel...")

# For multimodal Gemma3, layers are at model.language_model.layers
# Override repeng's layer detection by setting repeng_layers attribute
if hasattr(base_model, 'language_model') and hasattr(base_model.language_model, 'layers'):
    print(f"Detected multimodal Gemma3 architecture")
    print(f"Language model layers found at: model.language_model.layers")
    base_model.repeng_layers = base_model.language_model.layers
    num_layers = len(base_model.language_model.layers)
    
    # CRITICAL FIX #2: Gemma3Config is missing num_hidden_layers attribute
    # repeng needs this for training, so we add it manually
    base_model.config.num_hidden_layers = num_layers
    print(f"Set config.num_hidden_layers = {num_layers}")
    
elif hasattr(base_model, 'model') and hasattr(base_model.model, 'layers'):
    print(f"Detected standard architecture")
    base_model.repeng_layers = base_model.model.layers
    num_layers = len(base_model.model.layers)
else:
    raise ValueError("Could not find model layers!")

print(f"Total layers: {num_layers}")

# Using layers -5 to -18 (last 13 layers, counting from the end)
layer_ids = list(range(-5, -18, -1))
print(f"Wrapping layers: {layer_ids}")

model = ControlModel(base_model, layer_ids)

# Convert negative indices to verify
actual_layer_ids = [i if i >= 0 else num_layers + i for i in layer_ids]
print(f"Actual layer indices: {actual_layer_ids}")
print(f"✓ ControlModel initialized successfully!")
print("Model ready for training steering vectors!")

## 3. Load Training Data

In [ ]:
# Load truncated output suffixes (conversation continuations)
with open("data/datagen/all_truncated_outputs.json") as f:
    output_suffixes = json.load(f)

print(f"Loaded {len(output_suffixes)} output suffixes for training")
print(f"Examples: {output_suffixes[:5]}")

In [ ]:
# Helper function to create datasets without truncation
def make_dataset(template: str, pos_personas: list[str], neg_personas: list[str], suffixes: list[str]):
    """
    Create a dataset of positive/negative pairs for training control vectors.
    
    Uses each suffix exactly once without any truncation or filtering.
    
    Args:
        template: String template with {persona} placeholder
        pos_personas: List of positive persona descriptions
        neg_personas: List of negative persona descriptions (same length as pos_personas)
        suffixes: List of text suffixes to use as-is
    
    Returns:
        List of DatasetEntry objects
    """
    dataset = []
    
    for suffix in suffixes:
        # Create positive/negative pairs for each persona
        for positive_persona, negative_persona in zip(pos_personas, neg_personas):
            positive_text = template.format(persona=positive_persona) + " " + suffix
            negative_text = template.format(persona=negative_persona) + " " + suffix
            
            dataset.append(
                DatasetEntry(
                    positive=positive_text,
                    negative=negative_text,
                )
            )
    
    return dataset

print("Dataset creation helper function defined.")

## 4. Train Primary ToM Vector (BEST ONE)

This is the recommended general-purpose ToM steering vector from the documentation.

**Note on Chat Templates:** For instruction-tuned models like Gemma-3-4B-IT, you can optionally use the model's chat template format. The simple template below works well, but you can also use `tokenizer.apply_chat_template()` for more structured conversations.

In [ ]:
# Create the general ToM dataset with chat formatting
# This contrasts good vs bad theory of mind capabilities

# APPROACH 1: Simple template (recommended for this use case)
tom_dataset = make_dataset(
    "Pretend you're someone who is {persona}.",  # Natural phrasing for instruction
    ["excellent at understanding minds, predicting behavior, and inferring what others think and feel"],
    ["terrible at understanding minds, predicting behavior, and inferring what others think and feel"],
    output_suffixes
)

print(f"Created ToM dataset with {len(tom_dataset)} training pairs")
print(f"\nExample pairs (first 3):")
for i in range(min(3, len(tom_dataset))):
    print(f"\n[{i}] Positive: {tom_dataset[i].positive[:150]}...")
    print(f"[{i}] Negative: {tom_dataset[i].negative[:150]}...")

# APPROACH 2 (Alternative): Use tokenizer.apply_chat_template for formal chat format
# Uncomment the code below if you want to use the official Gemma chat template:
#
# def make_chat_dataset(user_prompt_template, pos_personas, neg_personas, suffixes):
#     dataset = []
#     for suffix in suffixes:
#         tokens = tokenizer.tokenize(suffix)
#         max_i = max(1, len(tokens) - 5)
#         for i in range(1, max_i + 1):
#             truncated = tokenizer.convert_tokens_to_string(tokens[:i])
#             for pos, neg in zip(pos_personas, neg_personas):
#                 # Create chat-formatted prompts
#                 pos_chat = tokenizer.apply_chat_template(
#                     [{"role": "user", "content": user_prompt_template.format(persona=pos)}],
#                     add_generation_prompt=True,
#                     tokenize=False
#                 )
#                 neg_chat = tokenizer.apply_chat_template(
#                     [{"role": "user", "content": user_prompt_template.format(persona=neg)}],
#                     add_generation_prompt=True,
#                     tokenize=False
#                 )
#                 dataset.append(DatasetEntry(
#                     positive=pos_chat + " " + truncated,
#                     negative=neg_chat + " " + truncated
#                 ))
#     return dataset
#
# tom_dataset = make_chat_dataset(
#     "Pretend you're someone who is {persona}.",
#     ["excellent at understanding minds, predicting behavior, and inferring what others think and feel"],
#     ["terrible at understanding minds, predicting behavior, and inferring what others think and feel"],
#     output_suffixes
# )

In [ ]:
# Train the general ToM vector
print("Training general ToM steering vector...")
print("This may take a few minutes...\n")

model.reset()  # Always reset before training
tom_vector = ControlVector.train(model, tokenizer, tom_dataset)

print("\nTraining complete!")
print(f"Vector contains directions for {len(tom_vector.directions)} layers")

In [ ]:
# Export the vector
vector_path = "tom_general.gguf"
tom_vector.export_gguf(vector_path)
print(f"Exported to: {vector_path}")

# Also save to Google Drive (if mounted)
try:
    import shutil
    drive_path = f"/content/drive/MyDrive/tom_steering_vectors/{vector_path}"
    shutil.copy(vector_path, drive_path)
    print(f"Also saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

### 4.1 Quick Vector Validation Test

Let's test if the vector actually works by comparing positive vs. negative steering on a simple conversational prompt.

In [ ]:
# Quick validation: Train a simple positive-negative vector using proper chat template
# Following the emotion.ipynb approach from repeng examples

print("Training a simple positive-negative steering vector for validation...")
print("Using proper chat template format.\n")

# Helper function to create properly formatted training pairs
def make_chat_dataset(instruction_template: str, pos_personas: list[str], neg_personas: list[str], suffixes: list[str]):
    """
    Create dataset using chat template format: 
    [USER INSTRUCTION with persona] [ASSISTANT RESPONSE START] suffix
    
    This follows the repeng emotion.ipynb approach.
    """
    dataset = []
    for suffix in suffixes:
        # Tokenize and use truncated versions (increases diversity)
        tokens = tokenizer.tokenize(suffix)
        for i in range(1, len(tokens)):
            truncated = tokenizer.convert_tokens_to_string(tokens[:i])
            for pos_persona, neg_persona in zip(pos_personas, neg_personas):
                # Format: <start_of_turn>user\nInstruction<end_of_turn>\n<start_of_turn>model\nSuffix
                pos_messages = [{"role": "user", "content": instruction_template.format(persona=pos_persona)}]
                neg_messages = [{"role": "user", "content": instruction_template.format(persona=neg_persona)}]
                
                # Apply chat template and add suffix as assistant response start
                pos_text = tokenizer.apply_chat_template(pos_messages, add_generation_prompt=True, tokenize=False) + truncated
                neg_text = tokenizer.apply_chat_template(neg_messages, add_generation_prompt=True, tokenize=False) + truncated
                
                dataset.append(DatasetEntry(positive=pos_text, negative=neg_text))
    return dataset

# Create validation dataset with clear positive/negative contrast
validation_dataset = make_chat_dataset(
    "Act as if you're extremely {persona}.",
    ["happy", "joyful", "ecstatic"],  # positive personas
    ["sad", "miserable", "depressed"],  # negative personas
    output_suffixes
)

print(f"Created validation dataset with {len(validation_dataset)} pairs")
print(f"\nExample training pair:")
print(f"Positive: {validation_dataset[0].positive[:150]}...")
print(f"Negative: {validation_dataset[0].negative[:150]}...\n")

# Train the validation vector
print("Training vector (this may take a few minutes)...")
model.reset()
pos_neg_vector = ControlVector.train(model, tokenizer, validation_dataset)
print("Training complete!\n")

# Test it on a simple prompt with proper chat formatting
def generate_with_chat_template(user_message: str, model, tokenizer, max_new_tokens=100):
    """Generate text using proper chat template."""
    messages = [{"role": "user", "content": user_message}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    output = model.generate(
        input_ids.input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

test_prompt = "How was your day?"

print("="*80)
print(f"Testing on: '{test_prompt}'")
print("="*80)

# Baseline
print("\n[BASELINE - No Steering]")
model.reset()
baseline = generate_with_chat_template(test_prompt, model, tokenizer, max_new_tokens=80)
print(baseline)

# Positive steering
print("\n" + "="*80)
print("[POSITIVE STEERING - coeff=+2.0]")
print("Expected: Happy, optimistic response")
print("-"*80)
model.set_control(pos_neg_vector, coeff=2.0)
positive = generate_with_chat_template(test_prompt, model, tokenizer, max_new_tokens=80)
print(positive)

# Negative steering
print("\n" + "="*80)
print("[NEGATIVE STEERING - coeff=-2.0]")
print("Expected: Sad, pessimistic response")
print("-"*80)
model.set_control(pos_neg_vector, coeff=-2.0)
negative = generate_with_chat_template(test_prompt, model, tokenizer, max_new_tokens=80)
print(negative)

model.reset()
print("\n" + "="*80)
print("\n✓ If you see clear sentiment differences, the vector training is working!")
print("  You can now proceed with confidence to train ToM vectors.")

## 5. Optional: Train Specialized ToM Vectors

These vectors target specific Theory of Mind skills. Uncomment any section to train that vector.

### 5.1 Order Init Vector (Implicit vs. Explicit Belief)
Trains the model to differentiate between inferring beliefs from context (implicit) and tracking explicitly stated beliefs.

In [ ]:
# Train Order Init vector
order_init_dataset = make_dataset(
    "Act as if you {persona}.",
    [
        "are exceptionally skilled at inferring an agent's beliefs from subtle contextual cues and their perspective",
        "can adeptly take an agent's perspective to understand their implicit knowledge"
    ],
    [
        "can only understand beliefs that are explicitly stated, struggling with context",
        "are unable to infer what an agent believes without a direct statement"
    ],
    output_suffixes
)

print(f"Training order_init vector ({len(order_init_dataset)} pairs)...")
model.reset()
order_init_vector = ControlVector.train(model, tokenizer, order_init_dataset)
order_init_vector.export_gguf("tom_order_init.gguf")
print("Exported to: tom_order_init.gguf")

# Save to Google Drive
try:
    import shutil
    drive_path = "/content/drive/MyDrive/tom_steering_vectors/tom_order_init.gguf"
    shutil.copy("tom_order_init.gguf", drive_path)
    print(f"Saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

### 5.2 Direction Vector (Forward vs. Backward Reasoning)
Focuses on temporal direction of reasoning: predicting future states vs. abducing past states.

In [ ]:
# Train Direction vector
direction_dataset = make_dataset(
    "You reason about events and mental states {persona}.",
    [
        "with exceptional foresight, predicting future beliefs and actions as a situation unfolds",
        "with sharp abductive skill, accurately reconstructing past beliefs from later observations"
    ],
    [
        "poorly, struggling to predict what will happen next based on the current situation",
        "with a lack of insight, unable to figure out past beliefs by looking at current outcomes"
    ],
    output_suffixes
)

print(f"Training direction vector ({len(direction_dataset)} pairs)...")
model.reset()
direction_vector = ControlVector.train(model, tokenizer, direction_dataset)
direction_vector.export_gguf("tom_direction.gguf")
print("Exported to: tom_direction.gguf")

# Save to Google Drive
try:
    import shutil
    drive_path = "/content/drive/MyDrive/tom_steering_vectors/tom_direction.gguf"
    shutil.copy("tom_direction.gguf", drive_path)
    print(f"Saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

### 5.3 Variable Vector (Belief vs. Action)
Distinguishes between representing what a person believes and predicting what they will do.

In [ ]:
# Train Variable vector
variable_dataset = make_dataset(
    "You are {persona} at distinguishing between an agent's beliefs and their actions.",
    [
        "highly skilled, accurately representing what an agent believes even when it's separate from what is true",
        "excellent, capably predicting an agent's actions based on their unique beliefs and goals"
    ],
    [
        "confused, often mixing up what someone thinks with what they do",
        "unable, failing to separate an agent's mental state from their subsequent behavior"
    ],
    output_suffixes
)

print(f"Training variable vector ({len(variable_dataset)} pairs)...")
model.reset()
variable_vector = ControlVector.train(model, tokenizer, variable_dataset)
variable_vector.export_gguf("tom_variable.gguf")
print("Exported to: tom_variable.gguf")

# Save to Google Drive
try:
    import shutil
    drive_path = "/content/drive/MyDrive/tom_steering_vectors/tom_variable.gguf"
    shutil.copy("tom_variable.gguf", drive_path)
    print(f"Saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

### 5.4 Belief Type Vector (True vs. False Belief)
Trains the model to handle both true and false belief scenarios.

In [ ]:
# Train Belief Type vector
belief_type_dataset = make_dataset(
    "You are {persona} at tracking what an agent believes.",
    [
        "adept, especially when an agent's belief is different from reality",
        "skilled, accurately simulating how an agent with a false belief perceives a situation"
    ],
    [
        "poor, always assuming an agent's beliefs align with the true state of the world",
        "unskilled, unable to comprehend that someone can hold a mistaken belief"
    ],
    output_suffixes
)

print(f"Training belief_type vector ({len(belief_type_dataset)} pairs)...")
model.reset()
belief_type_vector = ControlVector.train(model, tokenizer, belief_type_dataset)
belief_type_vector.export_gguf("tom_belief_type.gguf")
print("Exported to: tom_belief_type.gguf")

# Save to Google Drive
try:
    import shutil
    drive_path = "/content/drive/MyDrive/tom_steering_vectors/tom_belief_type.gguf"
    shutil.copy("tom_belief_type.gguf", drive_path)
    print(f"Saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

### 5.5 Forward Belief Vector (True & False)
Tracks an agent's beliefs as events unfold, including the formation of false beliefs.

In [ ]:
# Train Forward Belief vector
forward_belief_dataset = make_dataset(
    "You track what people believe as events happen {persona}.",
    [
        "with high accuracy, updating their beliefs based on new information they receive",
        "exceptionally well, recognizing when an agent has missed information and therefore holds a false belief"
    ],
    [
        "incorrectly, assuming everyone knows everything as it happens",
        "poorly, struggling to understand that a person's knowledge is limited to what they have observed"
    ],
    output_suffixes
)

print(f"Training forward_belief vector ({len(forward_belief_dataset)} pairs)...")
model.reset()
forward_belief_vector = ControlVector.train(model, tokenizer, forward_belief_dataset)
forward_belief_vector.export_gguf("tom_forward_belief.gguf")
print("Exported to: tom_forward_belief.gguf")

# Save to Google Drive
try:
    import shutil
    drive_path = "/content/drive/MyDrive/tom_steering_vectors/tom_forward_belief.gguf"
    shutil.copy("tom_forward_belief.gguf", drive_path)
    print(f"Saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

### 5.6 Backward Belief Vector (True & False)
Infers an agent's prior beliefs (both true and false) by observing later outcomes.

In [ ]:
# Train Backward Belief vector
backward_belief_dataset = make_dataset(
    "You infer past beliefs from current evidence {persona}.",
    [
        "skillfully, reconstructing what someone must have believed by looking at their later actions",
        "with excellence, determining if an outcome was due to a prior false belief or a change in the world"
    ],
    [
        "poorly, unable to work backward from an action to understand the belief that caused it",
        "inaccurately, failing to distinguish between ignorance and a changed reality when explaining events"
    ],
    output_suffixes
)

print(f"Training backward_belief vector ({len(backward_belief_dataset)} pairs)...")
model.reset()
backward_belief_vector = ControlVector.train(model, tokenizer, backward_belief_dataset)
backward_belief_vector.export_gguf("tom_backward_belief.gguf")
print("Exported to: tom_backward_belief.gguf")

# Save to Google Drive
try:
    import shutil
    drive_path = "/content/drive/MyDrive/tom_steering_vectors/tom_backward_belief.gguf"
    shutil.copy("tom_backward_belief.gguf", drive_path)
    print(f"Saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

### 5.7 Forward Action Vector (True & False Belief)
Predicts an agent's actions based on their goals and their (potentially false) beliefs.

In [ ]:
# Train Forward Action vector
forward_action_dataset = make_dataset(
    "You are {persona} at predicting what people will do.",
    [
        "excellent, able to simulate the plan an agent will follow based on their unique beliefs about the world",
        "adept, especially at predicting actions that logically follow from an agent's mistaken beliefs"
    ],
    [
        "poor, only able to predict actions based on the actual state of the world while ignoring individual beliefs",
        "unskilled, assuming that people always act with perfect and complete information"
    ],
    output_suffixes
)

print(f"Training forward_action vector ({len(forward_action_dataset)} pairs)...")
model.reset()
forward_action_vector = ControlVector.train(model, tokenizer, forward_action_dataset)
forward_action_vector.export_gguf("tom_forward_action.gguf")
print("Exported to: tom_forward_action.gguf")

# Save to Google Drive
try:
    import shutil
    drive_path = "/content/drive/MyDrive/tom_steering_vectors/tom_forward_action.gguf"
    shutil.copy("tom_forward_action.gguf", drive_path)
    print(f"Saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

### 5.8 Core Capabilities Vector (General Theory of Mind)
A broad set of Theory of Mind skills including perspective-taking and causal-temporal reasoning.

In [ ]:
# Train Core Capabilities vector
core_tom_dataset = make_dataset(
    "Act as if you have {persona}.",
    [
        "an exceptional ability to take others' perspectives and simulate their mental states",
        "a strong capacity for counterfactual reasoning about what others believe to be true"
    ],
    [
        "a very limited ability to understand how others think and feel",
        "a poor capacity for inhibiting your own knowledge to understand another's perspective"
    ],
    output_suffixes
)

print(f"Training core_tom vector ({len(core_tom_dataset)} pairs)...")
model.reset()
core_tom_vector = ControlVector.train(model, tokenizer, core_tom_dataset)
core_tom_vector.export_gguf("tom_core_capabilities.gguf")
print("Exported to: tom_core_capabilities.gguf")

# Save to Google Drive
try:
    import shutil
    drive_path = "/content/drive/MyDrive/tom_steering_vectors/tom_core_capabilities.gguf"
    shutil.copy("tom_core_capabilities.gguf", drive_path)
    print(f"Saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

### 5.9 Combined Specialized Vector
Combines all 7 specialized ToM vectors into a single comprehensive vector.

In [ ]:
# Combine all 7 specialized vectors
# (Make sure you've run all 7 specialized vector sections above first!)

combined_tom_vector = (
    order_init_vector +
    direction_vector + 
    variable_vector +
    belief_type_vector +
    forward_belief_vector +
    backward_belief_vector +
    forward_action_vector
) / 7  # Average the vectors

combined_tom_vector.export_gguf("tom_combined_specialized.gguf")
print("Exported combined vector to: tom_combined_specialized.gguf")

# Save to Google Drive
try:
    import shutil
    drive_path = "/content/drive/MyDrive/tom_steering_vectors/tom_combined_specialized.gguf"
    shutil.copy("tom_combined_specialized.gguf", drive_path)
    print(f"Saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

## 6. Test the ToM Vector

In [ ]:
# Helper function for generation
def generate_text(prompt, model, tokenizer, max_new_tokens=128):
    """
    Generate text from the model.
    """
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # Deterministic
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1
    )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Test prompt: A classic false belief scenario
test_prompt = """Sarah puts her toy in the red box and leaves the room. 
While she's gone, John moves the toy to the blue box. 
When Sarah returns, where will she look for her toy?

Answer:"""

print("Testing ToM steering vector...\n")
print("="*80)

# Baseline (no steering)
print("\n[BASELINE - No Steering]")
model.reset()
baseline_output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(baseline_output)

# With positive ToM steering
print("\n" + "="*80)
print("\n[WITH ToM STEERING - Strength: 1.5]")
model.set_control(tom_vector, coeff=1.5)
steered_output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(steered_output)

# With negative ToM steering (anti-ToM)
print("\n" + "="*80)
print("\n[ANTI-ToM STEERING - Strength: -2.0]")
model.set_control(tom_vector, coeff=-2.0)
anti_steered_output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(anti_steered_output)

# Reset model
model.reset()
print("\n" + "="*80)

### Test with Your Own Prompts

In [ ]:
# Try your own ToM scenario
custom_prompt = """John thinks the meeting is at 3pm, but it was changed to 2pm. 
He wasn't informed about the change. What time will John show up?

Answer:"""

print("Custom test:")
print("="*80)
model.set_control(tom_vector, coeff=1.5)
result = generate_text(custom_prompt, model, tokenizer, max_new_tokens=100)
print(result)
model.reset()

## 7. Download Vectors

Download the trained vectors to your local machine.

In [ ]:
from google.colab import files
import os

# List all .gguf files in current directory
gguf_files = [f for f in os.listdir('.') if f.endswith('.gguf')]

print(f"Found {len(gguf_files)} vector file(s):")
for f in gguf_files:
    print(f"  - {f}")

# Download each file
print("\nDownloading...")
for f in gguf_files:
    files.download(f)
    print(f"Downloaded: {f}")

print("\nAll vectors downloaded!")

## 8. How to Use the Vectors Later

To use these vectors in future sessions:

In [ ]:
# # Example: Load and use a saved vector
# from repeng import ControlVector, ControlModel
# from transformers import AutoModelForCausalLM, AutoTokenizer
# 
# # Load model
# model = AutoModelForCausalLM.from_pretrained("google/gemma-3-4b-it")
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")
# model = ControlModel(model, list(range(-5, -18, -1)))
# 
# # Load the vector
# tom_vector = ControlVector.import_gguf("tom_general.gguf")
# 
# # Apply the vector
# model.set_control(tom_vector, coeff=1.5)
# 
# # Generate with ToM enhancement
# # ... your generation code ...
# 
# # Reset when done
# model.reset()

## Notes

**Vector Strength (coeff parameter):**
- Start with values between -2.5 and 2.5
- Positive values: Enhance ToM capabilities
- Negative values: Reduce ToM capabilities (useful for testing)
- Typical good range: 1.0 to 2.0

**Best Practices:**
- Always call `model.reset()` before training a new vector
- Always call `model.reset()` after generation if you want baseline behavior
- Export vectors immediately after training to avoid losing them
- Test different coefficient values to find what works best for your use case

**Memory Tips:**
- If you run out of memory, restart runtime and reduce batch size in training
- Use `torch.float16` (already configured) to save memory
- Consider using fewer layers in ControlModel if needed